# AI Software Development Team

```
                    USER REQUEST
                          │
                          ▼
                ┌─────────────────┐
                │ SUPERVISOR NODE │
                └─────────────────┘
                          │
          ┌───────────────┼────────────────┐
          ▼               ▼                ▼
 ┌────────────────┐ ┌──────────────┐ ┌──────────────┐
 │ RESEARCH AGENT │ │ ARCHITECT    │ │ HUMAN REVIEW │
 │                │ │ AGENT        │ │ (optional)   │
 └────────────────┘ └──────────────┘ └──────────────┘
          │                │
          ▼                ▼
  Documentation      System Design
  API Search          Planning
          │                │
          └────────┬───────┘
                   ▼
          ┌────────────────┐
          │ CODING AGENT   │
          └────────────────┘
                   │
                   ▼
           Tool Calling:
           - File writer
           - Dependency lookup
           - Code generator
                   │
                   ▼
          ┌────────────────┐
          │ REVIEW AGENT   │
          └────────────────┘
                   │
      ┌────────────┴────────────┐
      ▼                         ▼
 APPROVED                  NEEDS FIX
      │                         │
      ▼                         │
┌──────────────┐                │
│ TEST AGENT   │◄───────────────┘
└──────────────┘
      │
      ▼
SUCCESS / FAILURE
      │
      ▼
FINAL RESPONSE
```

In [ ]:
from rich import print
import os
from dotenv import load_dotenv

load_dotenv()

In [ ]:
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["TAVILY_API_KEY"]=os.getenv("TAVILY_API_KEY")
os.environ["LANGSMITH_API_KEY"]=os.getenv("LANGSMITH_API_KEY")

In [ ]:
from langchain.chat_models import init_chat_model

llm=init_chat_model(model="groq:meta-llama/llama-4-scout-17b-16e-instruct")

In [ ]:
from typing import TypedDict, Optional

class WorkflowState(TypedDict):
    user_request:str
    research_notes: list[str]
    architecture_plan: Optional[str]
    generated_code: Optional[str]
    review_feedback: Optional[str]
    review_approved: bool
    test_results: Optional[str]
    next_step: Optional[str]
    retry_count: int

#### Research Tools

In [ ]:
from langchain_core.tools import tool

In [ ]:
@tool
def search_docs(query:str)->str:
    """Search technical documentation"""

    docs={
        "oauth":"""
        OAuth Best Practices:
        - Use secure redirect URIs
        - Store tokens securely
        - Use CSRF protection
        - Use Authlib for FastAPI
        """,

        "fastapi oauth": """
        FastAPI OAuth Example:
        - Use Authlib
        - Configure callback routes
        - Use session middleware
        """
    }

    query=query.lower()

    for key, value in docs.items():
        if key in query:
            return value
        
    return "No Documentation found"

#### Architecture Tool

In [ ]:
@tool
def generate_architecture(requirement: str)->str:
    """Generate architecture plan."""

    return f"""
    Architecture plan for:

    {requirement}

    Components:
    1. OAuth Routes
    2. Session Middleware
    3. Callback Handlers
    4. Token Storage
    5. Frontend Login Flow
    """

#### Coding Tools

In [ ]:
@tool
def lookup_package(package_name: str)->str:
    """Lookup package information."""
    packages = {
        "authlib": "Authlib latest version supports FastAPI OAuth integrations.",
        "fastapi": "FastAPI supports async authentication routes."
    }
    return packages.get(package_name.lower(), "Package not found.")

In [ ]:
@tool
def write_code(requirement: str) -> str:
    """
    Generate implementation code.
    """

    return f'''
from fastapi import FastAPI
from authlib.integrations.starlette_client import OAuth

app = FastAPI()

oauth = OAuth()

@app.get("/login/google")
async def login_google():
    return {{"message": "Google OAuth Login"}}
'''

#### Review Tool

In [ ]:
@tool
def review_code_tool(code: str) -> str:
    """
    Review generated code.
    """

    if "csrf" not in code.lower():
        return "Missing CSRF protection."

    return "Code looks good."

#### Testing Tool

In [ ]:
@tool
def run_tests(code: str) -> str:
    """
    Simulate test execution.
    """

    if "FastAPI" in code:
        return "All tests passed."

    return "Tests failed."

#### Research Agent

In [ ]:
# from langchain.agents import create_agent
from langgraph.prebuilt import create_react_agent

In [ ]:
research_agent=create_react_agent(llm,tools=[search_docs])

#### Architect Agent

In [ ]:
architect_agent=create_react_agent(llm, tools=[generate_architecture])

#### Coding Agent

In [ ]:
coding_agent=create_react_agent(llm,tools=[lookup_package,write_code])

#### Review Agent

In [ ]:
review_agent=create_react_agent(llm, tools=[review_code_tool])

#### Testing Agent

In [ ]:
testing_agent=create_react_agent(llm, tools=[run_tests])

#### Supervisor Decision

In [ ]:
from pydantic import BaseModel
from typing import Literal

class SupervisorDecision(BaseModel):
    next_step: Literal[
        "research",
        "architect",
        "coder",
        "review",
        "tester",
        "finish"
    ]
    reasoning: str

#### Review Decision

In [ ]:
class ReviewDecision(BaseModel):

    approved: bool

    feedback: str

#### Supervisor Node

In [ ]:
def supervisor_node(state: WorkflowState):

    print("\n========== SUPERVISOR ==========")

    if not state["research_notes"]:
        next_step = "research"

    elif not state["architecture_plan"]:
        next_step = "architect"

    elif not state["generated_code"]:
        next_step = "coder"

    elif not state["review_feedback"]:
        next_step = "review"

    elif not state["review_approved"]:
        next_step = "coder"

    elif not state["test_results"]:
        next_step = "tester"

    else:
        next_step = "finish"

    print(f"Next Step: {next_step}")

    return {
        "next_step": next_step
    }

#### Research Node

In [ ]:
from langchain_core.messages import HumanMessage

In [ ]:
def research_node(state: WorkflowState):

    print("\n========== RESEARCH AGENT ==========")

    result = research_agent.invoke({
        "messages": [
            HumanMessage(
                content=f"""
                Research requirements for:

                {state["user_request"]}

                Use available tools.
                """
            )
        ]
    })

    output = result["messages"][-1].content

    print(output)

    return {
        "research_notes": [output]
    }

#### Architect Node

In [ ]:
def architect_node(state: WorkflowState):

    print("\n========== ARCHITECT AGENT ==========")

    result = architect_agent.invoke({
        "messages": [
            HumanMessage(
                content=f"""
                Create architecture plan.

                Requirement:
                {state["user_request"]}

                Research:
                {state["research_notes"]}
                """
            )
        ]
    })

    output = result["messages"][-1].content

    print(output)

    return {
        "architecture_plan": output
    }

#### Coding Node

In [ ]:
def coding_node(state: WorkflowState):

    print("\n========== CODING AGENT ==========")

    result = coding_agent.invoke({
        "messages": [
            HumanMessage(
                content=f"""
                Generate production-ready code.

                Requirement:
                {state["user_request"]}

                Architecture:
                {state["architecture_plan"]}

                Research:
                {state["research_notes"]}

                Include CSRF protection.
                """
            )
        ]
    })

    output = result["messages"][-1].content

    print(output)

    return {
        "generated_code": output,
        "review_feedback": None
    }

#### Review Node

In [ ]:
def review_node(state: WorkflowState):

    print("\n========== REVIEW AGENT ==========")

    result = review_agent.invoke({
        "messages": [
            HumanMessage(
                content=f"""
                Review this code carefully.

                {state["generated_code"]}
                """
            )
        ]
    })

    output = result["messages"][-1].content

    print(output)

    approved = "looks good" in output.lower()

    return {
        "review_feedback": output,
        "review_approved": approved
    }

#### Testing Node

In [ ]:
def testing_node(state: WorkflowState):

    print("\n========== TESTING AGENT ==========")

    result = testing_agent.invoke({
        "messages": [
            HumanMessage(
                content=f"""
                Run tests on this code.

                {state["generated_code"]}
                """
            )
        ]
    })

    output = result["messages"][-1].content

    print(output)

    return {
        "test_results": output
    }

#### Conditional Routing

In [ ]:
def supervisor_router(state: WorkflowState):

    return state["next_step"]

In [ ]:
from langgraph.graph import StateGraph, END

In [ ]:
graph = StateGraph(WorkflowState)

In [ ]:
graph.add_node("supervisor", supervisor_node)

graph.add_node("research", research_node)

graph.add_node("architect", architect_node)

graph.add_node("coder", coding_node)

graph.add_node("review", review_node)

graph.add_node("tester", testing_node)

In [ ]:
graph.set_entry_point("supervisor")

In [ ]:
graph.add_conditional_edges(
    "supervisor",
    supervisor_router,
    {
        "research": "research",
        "architect": "architect",
        "coder": "coder",
        "review": "review",
        "tester": "tester",
        "finish": END
    }
)

In [ ]:
graph.add_edge("research", "supervisor")

graph.add_edge("architect", "supervisor")

graph.add_edge("coder", "supervisor")

graph.add_edge("review", "supervisor")

graph.add_edge("tester", "supervisor")

In [ ]:
app = graph.compile()

In [ ]:
app

In [ ]:
initial_state = {

    "user_request": """
    Build a FastAPI Google OAuth login system.
    """,

    "research_notes": [],

    "architecture_plan": None,

    "generated_code": None,

    "review_feedback": None,

    "review_approved": False,

    "test_results": None,

    "next_step": None,

    "retry_count": 0
}

In [ ]:
final_state = app.invoke(initial_state)